# 🗺️ Geo-AI Training — Workshop 1: จำแนกพื้นที่จาก Orthomosaic test 1

**แนวคิด**: จำแนกพื้นที่ 2 แบบเปรียบเทียบกัน
1. **Unsupervised (KMeans)** — แบ่งพิกเซลที่มีสีคล้ายกันออกเป็นกลุ่ม โดยไม่ต้องมี label
2. **Supervised (Random Forest)** — เทรนด้วยข้อมูล label ที่เตรียมไว้ (building, water, bareland, road, plant) เพื่อให้ผลจำแนกมีความหมายชัดเจนและแม่นยำกว่า

## 🧩 Setup



> 📦 ก่อนรัน ติดตั้ง dependencies ให้ครบ
> ```
> pip install -r requirements.txt
> ```

In [ ]:
# ติดตั้ง library ที่ใช้ในไฟล์นี้ (ใช้เวลาสักครู่ตอนรันครั้งแรก)
!pip install -q rasterio scikit-learn geopandas shapely folium gdown

print("ติดตั้งเสร็จแล้ว ✅")

### ติดตั้งฟอนต์ไทย (สำหรับกราฟที่มีข้อความไทย)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request
import os

# โหลดฟอนต์ Sarabun (ฟอนต์ไทยจาก Google Fonts) มาใช้กับกราฟ — โหลดครั้งเดียว ถ้ามีไฟล์แล้วข้ามได้เลย
font_path = "Sarabun-Regular.ttf"
if not os.path.exists(font_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf",
        font_path
    )
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "Sarabun"
plt.rcParams["axes.unicode_minus"] = False

print("ติดตั้งฟอนต์ไทยเสร็จแล้ว ✅")

In [ ]:
import os

# 📁 ใช้พื้นที่เก็บไฟล์ชั่วคราวในเครื่อง Colab (ไม่เชื่อม Google Drive)
# ⚠️ ไฟล์ในนี้จะหายไปเมื่อ Colab runtime ถูกตัดการเชื่อมต่อ/รีสตาร์ท — ดาวน์โหลดเก็บเองก่อนปิดเครื่อง
# (ใช้แผง Files ด้านซ้ายของ Colab คลิกขวาไฟล์ > Download)
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/outputs"
EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("EXPORT_DIR :", EXPORT_DIR)

### โหลดข้อมูลจาก Google Drive

ดาวน์โหลดข้อมูลโดรนจริง 3 ไฟล์ (ถ้ามีอยู่แล้วใน `DATA_DIR` จะข้ามการโหลดซ้ำ)

In [ ]:
import gdown

FILES = {
    "orthophoto.tif": "1i3yprs-Es03CFbMKx2rTvcsXJs0KOX_r",
}

for filename, file_id in FILES.items():
    out_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(out_path):
        print(f"✅ มีไฟล์อยู่แล้ว: {filename}")
    else:
        print(f"⬇️  กำลังโหลด: {filename} ...")
        gdown.download(id=file_id, output=out_path, quiet=False)

print("\nเสร็จแล้ว พร้อมใช้งาน")

## Supervised — จำแนกพื้นที่ด้วย Random Forest

ใช้ข้อมูล label ที่เตรียมไว้ (polygon พร้อม field `class`: building, water, bareland, road, plant) มาเทรนโมเดล

In [ ]:
import rasterio
import numpy as np
from rasterio.enums import Resampling

# ตั้งค่าอัตราการย่อ (0.5 คือย่อครึ่งหนึ่ง, 0.25 คือย่อเหลือ 1 ใน 4)
upscale_factor = 0.5

with rasterio.open(os.path.join(DATA_DIR, "orthophoto.tif")) as src:
    # เก็บขนาดเดิมไว้เพื่อเปรียบเทียบ
    orig_h, orig_w = src.height, src.width

    # คำนวณขนาดใหม่
    new_height = int(orig_h * upscale_factor)
    new_width = int(orig_w * upscale_factor)

    # อ่านข้อมูลพร้อม Resampling
    ortho = src.read(
        out_shape=(src.count, new_height, new_width),
        resampling=Resampling.bilinear
    )

    # อัปเดต metadata ให้ตรงกับขนาดใหม่
    ortho_profile = src.profile
    ortho_profile.update({
        'height': new_height,
        'width': new_width,
        'transform': src.transform * src.transform.scale(
            (src.width / new_width),
            (src.height / new_height)
        )
    })

H, W = ortho.shape[1], ortho.shape[2]

# แปลงภาพเป็นรายการพิกเซล
pixels = ortho[:3].reshape(3, -1).T.astype(np.float32)

print(f"ขนาดภาพเดิม: {orig_h} x {orig_w} ({orig_h * orig_w:,} พิกเซล)")
print(f"ขนาดภาพหลัง Downsampling: {H} x {W} ({pixels.shape[0]:,} พิกเซล)")
print(f"ประมวลผลเร็วขึ้นประมาณ: {1/(upscale_factor**2):.0f} เท่า")
print("ระบบพิกัด (CRS):", ortho_profile["crs"])

In [ ]:
LABEL_ID = "188PNVpV1s72BLbJVnbRk69M8swcD1gqr"
label_path = os.path.join(DATA_DIR, "land_use.geojson")


if not os.path.exists(label_path):
    gdown.download(id=LABEL_ID, output=label_path, quiet=False)

import geopandas as gpd

labels_gdf = gpd.read_file(label_path)


# การนำแนกพื้นที่
# w=water
# u=urban
# a=agri
# f=forest
# m=Miscellaneous

CLASS_NAMES = ["u", "w", "a", "m", "f"]
CLASS_COLORS = {
    "u": (150, 150, 150),
    "w": (60, 110, 180),
    "a": (180, 170, 120),
    "m": (90, 90, 90),
    "f": (40, 110, 50),
}
class_to_id = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}  # 0 = ไม่มี label

print("จำนวน label ทั้งหมด:", len(labels_gdf))
print(labels_gdf["name"].value_counts())

In [ ]:
import matplotlib.patches as mpatches
from rasterio.transform import array_bounds

# ขอบเขตภาพในหน่วยพิกัดเดิม (west, south, east, north) ใช้กำหนดตำแหน่งวาดภาพให้ตรงกับ label
west, south, east, north = array_bounds(H, W, ortho_profile["transform"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(ortho.transpose(1, 2, 0), extent=(west, east, south, north))
for name in CLASS_NAMES:
    subset = labels_gdf[labels_gdf["name"] == name]
    if len(subset):
        subset.plot(ax=ax, facecolor=np.array(CLASS_COLORS[name]) / 255,
                    edgecolor="black", linewidth=0.8)

handles = [mpatches.Patch(color=np.array(c) / 255, label=n) for n, c in CLASS_COLORS.items()]
ax.legend(handles=handles, loc="upper right", fontsize=8, title="ประเภท")
ax.set_title("ตำแหน่งพื้นที่ Training (Label) บน Orthomosaic")
ax.axis("off")
plt.tight_layout()
plt.show()

### เตรียมข้อมูลเทรนจาก label

In [ ]:
from rasterio.features import rasterize

# วาด polygon label ลงบนภาพให้เป็นราสเตอร์ขนาดเดียวกับ Orthomosaic
shapes = [(geom, class_to_id[cls]) for geom, cls in zip(labels_gdf.geometry, labels_gdf["name"])
          if cls in class_to_id]
label_raster = rasterize(shapes, out_shape=(H, W), transform=ortho_profile["transform"],
                          fill=0, dtype="uint8")

# เตรียมข้อมูลเทรน: ใช้เฉพาะพิกเซลที่มี label
train_mask = label_raster > 0
X = pixels[train_mask.ravel()]
y = label_raster[train_mask]

print("จำนวนพิกเซลที่มี label ทั้งหมด:", train_mask.sum())

### แบ่งชุดเทรน/ชุดทดสอบ

แบ่งข้อมูล label ออกเป็นชุดเทรน (70%) และชุดทดสอบ (30%) ที่โมเดลไม่เคยเห็น เพื่อใช้วัดความแม่นยำอย่างเป็นธรรมภายหลัง (`stratify=y` เพื่อให้สัดส่วนแต่ละคลาสในสองชุดใกล้เคียงกัน)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print("จำนวนตัวอย่างเทรนต่อคลาส (จากชุดเทรนเท่านั้น):")
for name, cid in class_to_id.items():
    print(f"  {name}: {(y_train == cid).sum():,} พิกเซล")

### เทรน Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 🔧 ปรับได้: n_estimators = จำนวนต้นไม้ใน Random Forest
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("เทรนเสร็จแล้ว ✅")

### ผลลัพธ์การจำแนกทั้งภาพ

ทำนายทั้งภาพ (ที่ขนาดย่อแล้วจากด้านบน จึงเร็วและ noise น้อยกว่าทำที่ความละเอียดเต็ม) แล้วเทียบกับผล Unsupervised

In [ ]:
pred = rf.predict(pixels).reshape(H, W)

# แปลงผลลัพธ์เป็นภาพสี (ใช้สีที่มีความหมายจริง ไม่ใช่สีสุ่มแบบ KMeans) 123
rf_rgb = np.zeros((H, W, 3), dtype=np.uint8)
for name, cid in class_to_id.items():
    rf_rgb[pred == cid] = CLASS_COLORS[name]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(ortho.transpose(1, 2, 0)); axes[0].set_title("Orthomosaic")
axes[1].imshow(rf_rgb); axes[1].set_title("Supervised (Random Forest)")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

print("สัดส่วนพื้นที่แต่ละคลาส (Random Forest):")
for name, cid in class_to_id.items():
    pct = (pred == cid).mean() * 100
    print(f"  {name}: {pct:.1f}%")

> ⚠️ **ข้อควรระวังเรื่องการวัดความแม่นยำ**: เราแบ่งชุดเทรน/ทดสอบแบบสุ่ม "รายพิกเซล" ซึ่งพิกเซลข้างเคียงกันในโพลิกอนเดียวกันมีสีคล้ายกันมาก (spatial autocorrelation) ทำให้ค่า accuracy ที่ได้ค่อนข้าง "สูงเกินจริง" เมื่อเทียบกับการเอาโมเดลไปใช้กับพื้นที่ใหม่จริงๆ — งานวิจัย/งานจริงมักแบ่งชุดทดสอบ "รายโพลิกอน" แทน (เช่น เก็บบางโพลิกอนทั้งอันไว้เป็นชุดทดสอบ ไม่ให้พิกเซลในโพลิกอนเดียวกันหลุดไปอยู่ทั้งสองฝั่ง) เพื่อให้ accuracy สะท้อนการ generalize จริง

### วัดความแม่นยำ (Accuracy)

วัดผลด้วยชุดทดสอบที่โมเดลไม่เคยเห็นตอนเทรน (ห้ามใช้ชุดเทรนมาวัด จะได้ค่าที่สูงเกินจริง)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_test = rf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
print(f"✅ ความแม่นยำบนชุดทดสอบ (test accuracy): {test_acc*100:.1f}%\n")
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from IPython.display import display

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred_test, labels=list(class_to_id.values()), zero_division=0)

report_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support,
})
display(report_df.round(2))

report_df.set_index("class")[["precision", "recall", "f1_score"]].plot(
    kind="bar", figsize=(8, 4), ylim=(0, 1))
plt.title(f"ความแม่นยำแยกตามคลาส (test accuracy รวม {test_acc*100:.1f}%)")
plt.ylabel("คะแนน (0-1)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> 💡 **เปรียบเทียบ**: KMeans (unsupervised) เร็วและไม่ต้องเตรียม label แต่บอกความหมายจริงไม่ได้ — Random Forest (supervised) ต้องมี label ก่อน แต่ผลลัพธ์มีความหมายชัดเจนและมักแม่นยำกว่า เพราะเรียนรู้จากตัวอย่างจริงที่เรากำหนด

## 🌍 แผนที่ Interactive (Folium)

ก่อน export ลองดูผลลัพธ์ทั้งหมดซ้อนกันบนแผนที่แบบ interactive ที่เลื่อน/ซูม/คลิกดูรายละเอียด และเปิด-ปิดแต่ละ layer ได้:
- Orthomosaic (ย่อขนาดแล้วเพื่อความเร็ว)
- ข้อมูล label (พื้นที่ training)
- ผล Unsupervised (KMeans)
- ผล Supervised (Random Forest) แปลงเป็น **vector** พร้อมคำนวณขนาดพื้นที่แต่ละประเภทเป็น **ไร่**

In [ ]:
import folium
from rasterio.warp import transform_bounds

# แปลงขอบเขตภาพจากระบบพิกัดเดิม เป็น lat/lon (EPSG:4326) ที่ folium ใช้วางแผนที่
lon_min, lat_min, lon_max, lat_max = transform_bounds(
    ortho_profile["crs"], "EPSG:4326", west, south, east, north)
image_bounds = [[lat_min, lon_min], [lat_max, lon_max]]  # [[lat ต่ำสุด, lon ต่ำสุด], [lat สูงสุด, lon สูงสุด]]

m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2],
                zoom_start=17, tiles="CartoDB positron")
print("สร้างแผนที่ฐานเรียบร้อย ✅ — เพิ่ม layer ต่อในเซลล์ถัดไป")

In [ ]:
# layer ที่ 1: Orthomosaic
folium.raster_layers.ImageOverlay(
    image=ortho.transpose(1, 2, 0), bounds=image_bounds,
    name="Orthomosaic", opacity=1.0,
).add_to(m)


print("เพิ่ม layer Orthomosaic แล้ว ✅")

In [ ]:
# layer ที่ 2: ข้อมูล label (พื้นที่ training) — ต้อง reproject เป็น lat/lon ก่อนใส่ folium
labels_4326 = labels_gdf.to_crs(4326)

def label_style(feature):
    color = CLASS_COLORS.get(feature["properties"]["name"], (255, 0, 0))
    return {"fillColor": f"rgb{color}", "color": "black", "weight": 1, "fillOpacity": 0.6}

folium.GeoJson(
    labels_4326, name="Label (Training Area)", style_function=label_style,
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["ประเภท:"]), show=False,
).add_to(m)

print("เพิ่ม layer Label แล้ว ✅")

In [ ]:
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape as shp_shape
from shapely.ops import unary_union
from scipy import ndimage
import geopandas as gpd

id_to_name = {cid: name for name, cid in class_to_id.items()}

# ลบ noise แบบเกลือพริกไทยก่อนแปลงเป็น vector ด้วย median filter
# พิกเซลเดี่ยวๆ ที่ทำนายผิดปนจะถูกปรับให้เข้ากับพิกเซลข้างเคียงส่วนใหญ่
# ลดจำนวน polygon เล็กจิ๋วลงมหาศาล ทำให้ dissolve เร็วขึ้นมาก
print("กำลัง smooth ผลลัพธ์เพื่อลด noise...")
pred_smooth = ndimage.median_filter(pred, size=3)

# คำนวณพื้นที่ 1 พิกเซลจริง (หน่วยเดียวกับ CRS ของภาพ) ใช้เป็นเกณฑ์กรอง polygon เล็กจิ๋ว
pixel_area = abs(ortho_profile["transform"].a * ortho_profile["transform"].e)
min_area = pixel_area * 40   # กรอง polygon ที่เล็กกว่า 40 พิกเซลทิ้ง — ปรับตัวเลขนี้ได้ถ้าอยากกรองมาก/น้อยกว่านี้

# สร้าง list เพื่อเก็บข้อมูล
data = []

print("กำลังแปลง Raster เป็น Vector...")
for geom, val in rio_shapes(pred_smooth.astype(np.uint8), transform=ortho_profile["transform"]):
    if val in id_to_name:
        poly = shp_shape(geom)
        # กรองเฉพาะ polygon ที่มีพื้นที่มากกว่าเกณฑ์ที่อิงขนาดพิกเซลจริง (ลด noise/ลดจำนวนชิ้น)
        if poly.area > min_area:
            data.append({"class": id_to_name[val], "geometry": poly})

rf_gdf = gpd.GeoDataFrame(data, crs=ortho_profile["crs"])

print(f"รวมจำนวน Polygon ทั้งหมด: {len(rf_gdf):,} ชิ้น")
print("กำลัง Dissolve (รวมกลุ่มพื้นที่)... อาจใช้เวลาสักครู่")

rf_dissolved = rf_gdf.dissolve(by="class", as_index=False)

# แปลงเป็นหน่วยเมตร (UTM Zone 47N) เพื่อคำนวณพื้นที่
rf_dissolved_utm = rf_dissolved.to_crs("EPSG:32647")
rf_dissolved["area_m2"] = rf_dissolved_utm.geometry.area
rf_dissolved["area_rai"] = (rf_dissolved["area_m2"] / 1600).round(2)

print("คำนวณพื้นที่เสร็จสิ้น ✅")
display(rf_dissolved[["class", "area_rai"]])

In [ ]:
import folium
import cv2
from rasterio.warp import transform_bounds

# ย่อภาพเพิ่มเฉพาะตอนแสดงผลบนแผนที่ ถ้าภาพ (แม้ downsample แล้ว) ยังใหญ่เกินไป
# ป้องกัน error "Buffered data was truncated" — ไม่ย่อเกินความจำเป็น (ไม่ upscale ถ้าเล็กอยู่แล้ว)
MAP_DISPLAY_LONG_SIDE = 1500
display_scale = min(1.0, MAP_DISPLAY_LONG_SIDE / max(H, W))
if display_scale < 1.0:
    map_H, map_W = int(H * display_scale), int(W * display_scale)
    ortho_for_map = cv2.resize(
        ortho[:3].transpose(1, 2, 0), (map_W, map_H), interpolation=cv2.INTER_AREA)
    print(f"ย่อภาพสำหรับแผนที่เพิ่มเติม: {H}x{W} → {map_H}x{map_W}")
else:
    ortho_for_map = ortho[:3].transpose(1, 2, 0)
    print("ภาพเล็กพอแล้ว ไม่ต้องย่อเพิ่ม")

# แปลงขอบเขตภาพจากระบบพิกัดเดิม เป็น lat/lon (EPSG:4326) ที่ folium ใช้วางแผนที่
lon_min, lat_min, lon_max, lat_max = transform_bounds(
    ortho_profile["crs"], "EPSG:4326", west, south, east, north)
image_bounds = [[lat_min, lon_min], [lat_max, lon_max]]

m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2],
                zoom_start=17, tiles="CartoDB positron")

# layer 1: Orthomosaic — เปิดแสดงเป็นค่าเริ่มต้น
folium.raster_layers.ImageOverlay(
    image=ortho_for_map, bounds=image_bounds,   # 🔧 ใช้ภาพที่เช็คขนาดแล้วแทน ortho เต็ม
    name="Orthomosaic", opacity=1.0, show=True,
).add_to(m)

# layer 2: ข้อมูล label (พื้นที่ training) — ปิดไว้เป็นค่าเริ่มต้น เปิดดูได้ทีหลัง
labels_4326 = labels_gdf.to_crs(4326)

def label_style(feature):
    color = CLASS_COLORS.get(feature["properties"]["name"], (255, 0, 0))
    return {"fillColor": f"rgb{color}", "color": "black", "weight": 1, "fillOpacity": 0.6}

folium.GeoJson(
    labels_4326, name="Label (Training Area)", style_function=label_style,
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["ประเภท:"]),
    show=False,
).add_to(m)

# layer 3: ผล Random Forest (vector) — เปิดแสดงเป็นค่าเริ่มต้น
rf_4326 = rf_dissolved.to_crs(4326)

def rf_style(feature):
    color = CLASS_COLORS.get(feature["properties"]["class"], (255, 0, 0))
    return {"fillColor": f"rgb{color}", "color": "black", "weight": 0.5, "fillOpacity": 0.5}

folium.GeoJson(
    rf_4326, name="Supervised (Random Forest) - vector", style_function=rf_style,
    tooltip=folium.GeoJsonTooltip(fields=["class"], aliases=["ประเภท:"]),
    show=True,
).add_to(m)

# ตัวเปิด-ปิด layer — collapsed=False ให้กางเป็นเช็คบอกซ์ค้างไว้เลย ไม่ต้องคลิกเปิดก่อน
folium.LayerControl(collapsed=False).add_to(m)

m

## 💾 Export ผลลัพธ์

In [ ]:
out_profile = ortho_profile.copy()
out_profile.update(count=1, dtype="uint8")

# ผลจาก Random Forest (supervised) — ราสเตอร์
with rasterio.open(os.path.join(EXPORT_DIR, "landcover_rf.tif"), "w", **out_profile) as dst:
    dst.write(pred.astype(np.uint8), 1)

# ผลจาก Random Forest — vector พร้อมขนาดพื้นที่ (ไร่) ต่อประเภท ไว้เปิดใน QGIS
rf_dissolved.to_file(os.path.join(EXPORT_DIR, "landcover_rf_by_class.geojson"), driver="GeoJSON")

print("✅ Export แล้ว: landcover_rf.tif, landcover_rf_by_class.geojson")

---
✅ **จบ Workshop 1** — ไปต่อที่ `2_workshop2_chm_trees.ipynb`